# TakeMeter — Investment Discourse Quality Classifier

**Project:** AI201 Project 3  
**Model:** Fine-tuned `distilbert-base-uncased`  
**Labels:** `analysis`, `speculation`, `reaction`  
**Dataset:** 200 examples (66% real Reddit, 34% synthetic)

---

## ⚡ Before Running Anything
1. Go to **Runtime → Change runtime type → T4 GPU** → Save
2. Run cells **in order** — each section depends on the previous one
3. If your session disconnects, re-run Sections 1 and 2 before continuing
4. Add your Groq API key via the 🔑 Secrets panel before running Section 5

---

## Sections
- **Section 1:** Install dependencies & upload dataset
- **Section 2:** Load, split, and tokenize data
- **Section 3:** Fine-tune DistilBERT
- **Section 4:** Evaluate fine-tuned model
- **Section 5:** Zero-shot baseline (Groq) ← run this BEFORE Section 3
- **Section 6:** Compare results & export

---
## Section 1: Install Dependencies & Upload Dataset

In [ ]:
!pip install transformers datasets scikit-learn groq -q
print("✅ Libraries installed")

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import io

# ─────────────────────────────────────────────
# LABEL MAP
# ─────────────────────────────────────────────
LABEL2ID = {"analysis": 0, "speculation": 1, "reaction": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

print("Label map:", LABEL2ID)
print("\n📂 Upload takemeter_dataset.csv when prompted...")
print("   Expected: 200 rows | analysis=87, speculation=66, reaction=47")

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"\n✅ Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

# Validate expected size
assert len(df) >= 200, f"Expected 200+ rows, got {len(df)}"
assert set(df['label'].unique()) == {'analysis', 'speculation', 'reaction'}, \
    f"Unexpected labels: {df['label'].unique()}"
print("\n✅ Dataset validation passed")
df.head(3)

---
## Section 2: Load, Split & Tokenize

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

df['label_id'] = df['label'].map(LABEL2ID)
df = df[['text', 'label', 'label_id']].dropna().reset_index(drop=True)

# 70 / 15 / 15 stratified split
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['label_id']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label_id']
)

print(f"Split sizes (200 row dataset):")
print(f"  Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
print(f"\nTrain label distribution:")
print(train_df['label'].value_counts())
print(f"\nTest label distribution:")
print(test_df['label'].value_counts())

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=512
    )

def to_hf_dataset(df):
    ds = Dataset.from_dict({
        'text': df['text'].tolist(),
        'labels': df['label_id'].tolist()
    })
    return ds.map(tokenize, batched=True)

print("Tokenizing datasets...")
train_ds = to_hf_dataset(train_df)
val_ds   = to_hf_dataset(val_df)
test_ds  = to_hf_dataset(test_df)

dataset = DatasetDict({'train': train_ds, 'validation': val_ds, 'test': test_ds})
print("\n✅ Tokenization complete")
print(dataset)

---
## ⚠️ Run Section 5 (Baseline) BEFORE Section 3 (Fine-tuning)

Lock your baseline numbers on the test set before training begins.  
Scroll down to Section 5, run it, then come back here.

---
## Section 3: Fine-Tune DistilBERT

⏱️ Expected time: **8–15 minutes** on T4 GPU with 200-row dataset

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score
import torch

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
print(f"\n✅ Model loaded: {MODEL_NAME}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# ─────────────────────────────────────────────
# Hyperparameters — rationale documented in README:
# lr=2e-5 retained (not increased): dataset contains mix of real Reddit
# posts (noisy) and synthetic posts (clean). Higher LR risks overfitting
# to synthetic patterns. 3 epochs sufficient for 200 examples.
# ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir='./takemeter_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir='./logs',
    logging_steps=10,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("\n🚀 Starting fine-tuning...")
trainer.train()

---
## Section 4: Evaluate Fine-Tuned Model on Test Set

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import json

print("Running evaluation on test set...")
predictions_output = trainer.predict(test_ds)
logits = predictions_output.predictions
true_labels = predictions_output.label_ids
pred_labels = np.argmax(logits, axis=-1)

# Softmax confidence scores
probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
confidence_scores = probs.max(axis=-1)

overall_acc = accuracy_score(true_labels, pred_labels)
print(f"\n{'='*50}")
print(f"FINE-TUNED MODEL — TEST SET RESULTS")
print(f"{'='*50}")
print(f"Overall Accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)")

label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
report = classification_report(
    true_labels, pred_labels,
    target_names=label_names,
    output_dict=True
)
report_str = classification_report(true_labels, pred_labels, target_names=label_names)
print(f"\nPer-Class Metrics:")
print(report_str)

# Confusion matrix
cm = confusion_matrix(true_labels, pred_labels)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('TakeMeter — Fine-Tuned DistilBERT Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Confusion matrix saved as confusion_matrix.png")

In [ ]:
# ─────────────────────────────────────────────
# Sample classifications with confidence scores
# (required for README and demo video)
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("SAMPLE CLASSIFICATIONS")
print("="*50)

sample_indices = []
# One correct per label
for label_id in range(NUM_LABELS):
    correct = np.where((pred_labels == true_labels) & (true_labels == label_id))[0]
    if len(correct) > 0:
        sample_indices.append(int(correct[0]))

# One incorrect
incorrect = np.where(pred_labels != true_labels)[0]
if len(incorrect) > 0:
    sample_indices.append(int(incorrect[0]))

sample_indices = sample_indices[:5]
test_texts = test_df['text'].tolist()

samples = []
for i in sample_indices:
    true_l = ID2LABEL[true_labels[i]]
    pred_l = ID2LABEL[pred_labels[i]]
    conf = float(confidence_scores[i])
    flag = "✅" if true_l == pred_l else "❌"
    print(f"\n{flag} Example {len(samples)+1}")
    print(f"   Text:      {test_texts[i][:150].replace(chr(10),' ')}...")
    print(f"   True:      {true_l}")
    print(f"   Predicted: {pred_l}")
    print(f"   Confidence:{conf:.4f} ({conf*100:.1f}%)")
    samples.append({
        'text': test_texts[i][:300],
        'true_label': true_l,
        'predicted_label': pred_l,
        'confidence': round(conf, 4),
        'correct': true_l == pred_l
    })

In [ ]:
# ─────────────────────────────────────────────
# Wrong predictions — all of them
# ─────────────────────────────────────────────
from collections import Counter

wrong_indices = np.where(pred_labels != true_labels)[0]
print(f"\n{'='*50}")
print(f"WRONG PREDICTIONS ({len(wrong_indices)} total)")
print(f"{'='*50}")

wrong_examples = []
pairs = Counter(
    f"{ID2LABEL[true_labels[i]]} → {ID2LABEL[pred_labels[i]]}"
    for i in wrong_indices
)
print(f"Error pairs: {dict(pairs)}")

for i in wrong_indices:
    ex = {
        'text': test_texts[i][:400],
        'true_label': ID2LABEL[true_labels[i]],
        'predicted_label': ID2LABEL[pred_labels[i]],
        'confidence': round(float(confidence_scores[i]), 4)
    }
    wrong_examples.append(ex)
    print(f"\n  True: {ex['true_label']} | Predicted: {ex['predicted_label']} | Conf: {ex['confidence']}")
    print(f"  Text: {ex['text'][:200]}...")

---
## Section 5: Zero-Shot Baseline (Groq)

**Run this BEFORE Section 3 to lock baseline numbers first.**

Setup:
1. Click the 🔑 icon in the left sidebar
2. Add secret named `GROQ_API_KEY` with your key
3. Enable notebook access

In [ ]:
from groq import Groq
from google.colab import userdata
import time

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✅ Groq API key loaded from Colab Secrets")
except:
    GROQ_API_KEY = input("Enter your Groq API key: ").strip()
    print("✅ Groq API key entered manually")

client = Groq(api_key=GROQ_API_KEY)

CLASSIFICATION_PROMPT = """\
You are a classifier for investment community posts. Classify the following post into exactly one of these three labels:

- analysis: A post that makes a structured investment argument backed by specific, verifiable financial data (earnings figures, P/E ratios, FCF, revenue growth, margins). The reasoning stands independently of the conclusion.
- speculation: A bold prediction or opinion about a stock or market with little or no supporting evidence. May cite one surface-level stat but the argument collapses without the conclusion.
- reaction: An immediate emotional response to a specific market event (earnings miss, Fed decision, price crash). Expresses a feeling in the moment, not an investment case.

Post:
{post_text}

Respond with exactly one word: analysis, speculation, or reaction."""

def classify_with_groq(text, max_retries=3):
    prompt = CLASSIFICATION_PROMPT.format(post_text=text[:2000])
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=10,
                temperature=0
            )
            raw = response.choices[0].message.content.strip().lower()
            for label in ['analysis', 'speculation', 'reaction']:
                if label in raw:
                    return label
            print(f"  ⚠️ Unparseable: '{raw}'")
            return None
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"  ❌ Failed after {max_retries} attempts: {e}")
                return None

test_texts_list = test_df['text'].tolist()
test_true_labels = test_df['label'].tolist()

print(f"\nClassifying {len(test_texts_list)} test examples with Groq...")
print("Expected time: 3–6 minutes\n")

baseline_predictions = []
unparseable_count = 0

for i, text in enumerate(test_texts_list):
    pred = classify_with_groq(text)
    baseline_predictions.append(pred)
    if pred is None:
        unparseable_count += 1
    if (i + 1) % 5 == 0:
        done = (i+1)/len(test_texts_list)*100
        print(f"  {i+1}/{len(test_texts_list)} ({done:.0f}%)")
    time.sleep(0.3)

print(f"\n✅ Done. Unparseable: {unparseable_count}")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

valid_pairs = [
    (true, pred)
    for true, pred in zip(test_true_labels, baseline_predictions)
    if pred is not None
]
baseline_true = [p[0] for p in valid_pairs]
baseline_pred = [p[1] for p in valid_pairs]

baseline_acc = accuracy_score(baseline_true, baseline_pred)

print(f"{'='*50}")
print(f"GROQ ZERO-SHOT BASELINE — TEST SET RESULTS")
print(f"{'='*50}")
print(f"Evaluated: {len(valid_pairs)}/{len(test_texts_list)} examples")
print(f"Overall Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
print(f"\nPer-Class Metrics:")
baseline_report_str = classification_report(baseline_true, baseline_pred)
baseline_report = classification_report(baseline_true, baseline_pred, output_dict=True)
print(baseline_report_str)
print("\n⬆️ Now go run Section 3 (fine-tuning) and Section 4 (evaluation)")

---
## Section 6: Compare Results & Export

In [ ]:
print(f"{'='*60}")
print(f"MODEL COMPARISON SUMMARY")
print(f"{'='*60}")
print(f"Dataset: {len(df)} examples | Train={len(train_df)} Val={len(val_df)} Test={len(test_df)}")
print(f"{'='*60}")
print(f"{'Metric':<30} {'Baseline (Groq)':>15} {'Fine-tuned':>15}")
print("-"*60)
print(f"{'Overall Accuracy':<30} {baseline_acc*100:>14.2f}% {overall_acc*100:>14.2f}%")

for label in ['analysis', 'speculation', 'reaction']:
    if label in baseline_report and label in report:
        b_f1 = baseline_report[label]['f1-score']
        f_f1 = report[label]['f1-score']
        delta = f_f1 - b_f1
        arrow = "↑" if delta > 0 else "↓"
        print(f"  F1 [{label:<13}]   {b_f1:>14.4f}  {f_f1:>14.4f}  {arrow}{abs(delta):.4f}")

improvement = (overall_acc - baseline_acc) * 100
print(f"\nOverall improvement: {improvement:+.2f}%")
if improvement >= 10:
    print("✅ Exceeds the +10% target set in planning.md")
else:
    print("⚠️ Below the +10% target — investigate in evaluation report")

In [ ]:
import json

evaluation_results = {
    "fine_tuned_model": {
        "base_model": MODEL_NAME,
        "overall_accuracy": round(float(overall_acc), 4),
        "per_class_metrics": {
            label: {
                "precision": round(report[label]['precision'], 4),
                "recall":    round(report[label]['recall'], 4),
                "f1":        round(report[label]['f1-score'], 4),
                "support":   int(report[label]['support'])
            }
            for label in ['analysis', 'speculation', 'reaction']
        },
        "confusion_matrix": cm.tolist(),
        "wrong_predictions": wrong_examples,
        "sample_classifications": samples
    },
    "baseline_model": {
        "model": "groq/llama-3.3-70b-versatile",
        "overall_accuracy": round(float(baseline_acc), 4),
        "per_class_metrics": {
            label: {
                "precision": round(baseline_report[label]['precision'], 4),
                "recall":    round(baseline_report[label]['recall'], 4),
                "f1":        round(baseline_report[label]['f1-score'], 4),
                "support":   int(baseline_report[label]['support'])
            }
            for label in ['analysis', 'speculation', 'reaction']
            if label in baseline_report
        },
        "evaluated_examples": len(valid_pairs),
        "unparseable": unparseable_count
    },
    "dataset": {
        "total": len(df),
        "real_reddit_pct": 66,
        "synthetic_pct": 34,
        "train": len(train_df),
        "val": len(val_df),
        "test": len(test_df),
        "label_distribution": df['label'].value_counts().to_dict()
    },
    "training": {
        "epochs": 3,
        "learning_rate": 2e-5,
        "batch_size": 16
    }
}

with open('evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print("✅ evaluation_results.json saved")
print("\n📥 Downloading output files...")
files.download('evaluation_results.json')
files.download('confusion_matrix.png')
print("\n✅ All done!")
print("\nNext steps:")
print("1. Commit evaluation_results.json and confusion_matrix.png to GitHub")
print("2. Update README Evaluation Report section with new numbers")
print("3. Record demo video")

---
## After Running This Notebook

**Files to commit to GitHub:**
- `evaluation_results.json`
- `confusion_matrix.png`

**README sections to update with new numbers:**
- Overall Accuracy table (both models)
- Per-Class Metrics tables (both models)
- Confusion matrix image
- Wrong Predictions Analysis (from Section 4 output)
- Sample Classifications table (from Section 4 output)

**Demo video (3–5 min) must show:**
- 3–5 posts classified with label + confidence visible
- 1 correct prediction narrated with explanation
- 1 incorrect prediction narrated with explanation
- Brief walkthrough of evaluation report